# BCS ordinal v4: experimento de regularización

**Fotografía → YOLO entrenado en esta ejecución → grupa → EfficientNet-B0 → categoría BCS ordenada.**

Esta versión nueva conserva la detección y la asociación del experimento anterior. Cambia la estimación de BCS por un modelo de **umbrales acumulativos con score compartido y umbrales estrictamente ordenados**, inspirado en la formulación ordinal de CORAL. No utiliza una softmax multiclase ni una neurona de regresión MAE.

Si entrenamiento contiene las clases [2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75], una etiqueta 3.00 se representa como [1,1,1,0,0,0]: indica qué límites supera. Se minimiza entropía cruzada binaria de los seis umbrales. La salida principal es una categoría; se conserva además el valor esperado como diagnóstico continuo.

Todo se define en este notebook en orden. No se importan módulos del repositorio. El catálogo se obtiene exclusivamente de entrenamiento y se guarda con el modelo.

**Estado experimental:** la clasificación ordinal no garantiza mejorar la regresión. El test anterior ya fue observado: una nueva evaluación sobre esas mismas vacas es exploratoria, no una prueba final independiente. Primero elegir modelo y fase con validación; para una conclusión final se necesitan animales nuevos.

Referencias: [Cao et al., rank-consistent ordinal regression](https://arxiv.org/abs/1901.07884) y [BinaryCrossentropy de Keras](https://keras.io/api/losses/probabilistic_losses/). Esta implementación impone el orden mediante incrementos softplus; no pretende reproducir literalmente el código original de CORAL.

## Qué se observó antes de proponer esta versión

V3 tiene un mejor MAE anotado de validación 0.263889; su pipeline obtuvo 0.198276 en 29/36 fotos (48.28% exactas y 75.86% dentro de ±0.25). En esas mismas fotos el ordinal con caja anotada dio 0.215517. La regresión v2 obtuvo MAE pipeline 0.293458 en 29/36 fotos. Estos resultados son de validación, no de test.

Las curvas muestran una brecha: en la última época de ajuste fino MAE train 0.1611 frente a val 0.2708. Es compatible con sobreajuste, pero train usa dropout y aumentos; por eso añadimos una evaluación determinista del train y val con el mismo checkpoint. No atribuimos toda la brecha automáticamente a una causa.

| Cambio | V3 | V4 propuesto |
|---|---|---|
| Dropout de la cabeza | 0.20 | 0.35 |
| Optimizador | Adam | AdamW |
| Weight decay | No explícito | 0.0001 |
| Capas finales ajustables (excepto BN) | 40 | 20 |

Los umbrales ordinales y los parámetros de normalización se excluyen del decay. Mantenemos semilla, lotes, épocas 60/80, tasas y aumentos para limitar cambios. No hay EarlyStopping. Cambiar varios elementos prueba esta combinación, no identifica causalmente el aporte de cada uno. Puede mejorar o empeorar; selección y comparación se hacen con validación.

Referencia de implementación: [AdamW de Keras](https://keras.io/api/optimizers/adamw/).

## 1. Preparar Kaggle

Adjunta solamente el dataset COCO original con sus fotos y carpetas train/valid/test. No necesitas modelos ni outputs anteriores. Activa GPU e Internet para instalar dependencias y descargar los pesos iniciales de YOLO e ImageNet. Ejecuta de arriba abajo.

Por defecto `REUTILIZAR_YOLO=False`: entrenamos YOLO durante 80 épocas desde los pesos preentrenados `yolo26n.pt`, y después el clasificador ordinal durante 60 épocas de cabeza y 80 de ajuste fino. Ninguna fase utiliza EarlyStopping. El mejor checkpoint de cada fase se conserva según validación aunque el entrenamiento continúe.

Al terminar, descarga el ZIP generado y guarda una versión con outputs para conservar los modelos. El archivo .ipynb por sí solo no incluye los pesos. Si más adelante recuperas un detector anterior, puedes activar explícitamente la reutilización.


In [ ]:
# Instalamos únicamente Ultralytics si falta; mantenemos los frameworks de Kaggle.
import importlib.util
import importlib.metadata
import subprocess
import sys

ULTRALYTICS_VERSION = "8.4.0"
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        f"ultralytics=={ULTRALYTICS_VERSION}"
    ])
else:
    version = importlib.metadata.version("ultralytics")
    if version != ULTRALYTICS_VERSION:
        raise RuntimeError(
            f"Ultralytics instalado: {version}. Este experimento utiliza "
            f"{ULTRALYTICS_VERSION}; prepara esa versión antes de continuar."
        )

### Rutas y parámetros

Se crea `bcs_secuencial_v4_ordinal_regularizado`, separado de la regresión. Mantenemos recortes 224, aumentos y tasas de v2 para no cambiar simultáneamente el formato de datos. El catálogo BCS se derivará de entrenamiento. No hay pesos de clase ni sobremuestreo.

La partición sigue separada por vaca. Cada foto es una muestra y las métricas principales son por foto; no hay agregación por animal. El score ordinal y sus probabilidades no representan una calibración clínica.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import gc
import hashlib
import json
import math
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from IPython.display import display
import torch
from ultralytics import YOLO

SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# None busca automáticamente un único dataset COCO dentro de /kaggle/input.
DATOS = None
EXPERIMENTO = "bcs_secuencial_v4_ordinal_regularizado"
SALIDA = Path("/kaggle/working") / EXPERIMENTO
SALIDA.mkdir(parents=True, exist_ok=True)

PESOS_YOLO = "yolo26n.pt"  # También acepta una ruta a pesos adjuntos.
EPOCAS_YOLO = 80
TAM_YOLO = 640
LOTE_YOLO = 4
PACIENCIA_YOLO = 0  # Ultralytics: 0 desactiva la parada temprana.
REUTILIZAR_YOLO = False
DETECTOR_PREVIO = None  # "/kaggle/working/bcs_secuencial_v2_regresion/detector"
if REUTILIZAR_YOLO and DETECTOR_PREVIO is None:
    opciones = []
    local = Path("/kaggle/working/bcs_secuencial_v2_regresion/detector")
    if (local / "weights/best.pt").is_file() and (local / "results.csv").is_file():
        opciones.append(local)
    for seleccion in Path("/kaggle/input").rglob("seleccion_regresor.json"):
        candidato = seleccion.parent / "detector"
        if (candidato / "weights/best.pt").is_file() and (candidato / "results.csv").is_file():
            opciones.append(candidato)
    opciones = sorted(set(opciones))
    assert len(opciones) == 1, (
        f"Indica DETECTOR_PREVIO con la carpeta detector entrenada de v2. Candidatas: {opciones}. "
        "Adjunta los outputs de entrenamiento; el ZIP de test no incluye pesos."
    )
    DETECTOR_PREVIO = str(opciones[0])
if REUTILIZAR_YOLO:
    assert DETECTOR_PREVIO is not None
else:
    DETECTOR_PREVIO = None

TAM_RECORTE = 224
LOTE_ORDINAL = 8
EPOCAS_CABEZA = 60
EPOCAS_AJUSTE = 80
LR_CABEZA = 3e-4
LR_AJUSTE = 5e-6
CAPAS_AJUSTE = 20
DROPOUT = 0.35
WEIGHT_DECAY = 1e-4
FACTOR_REDUCCION_LR = 0.5
PACIENCIA_LR = 8  # Reduce la tasa; no detiene el entrenamiento.
LR_MINIMA = 1e-7
PESOS_EFFICIENTNET = "imagenet"

CONFIANZA = 0.25
COBERTURA_GRUPA = 0.8
TOLERANCIA_CAJA = 0.02  # Píxeles; admite pequeños errores de redondeo COCO.
PARTICION_EVALUACION = "val"
TEST_FINAL_CONFIRMADO = False
IMAGEN_NUEVA = None

assert torch.cuda.is_available(), "Activa una GPU en Kaggle antes de entrenar."
print("GPU del detector:", torch.cuda.get_device_name(0))
print("Resultados:", SALIDA)

## 2. Localizar y conocer el dataset

COCO guarda por separado las imágenes, las categorías y las cajas. Una caja tiene el formato `[x, y, ancho, alto]`, en píxeles. Mostramos primero las cantidades y categorías originales, antes de transformarlas.

La categoría contenedora de Roboflow puede aparecer en la lista de categorías; solo `cow` y `ass` se consideran objetos entrenables. `valid` se llamará `val` en nuestros resultados, pero conservamos las particiones originales.

In [ ]:
if DATOS is None:
    candidatas = sorted({
        p.parent.parent
        for p in Path("/kaggle/input").rglob("_annotations.coco.json")
        if p.parent.name == "train"
        and (p.parent.parent / "valid/_annotations.coco.json").is_file()
        and (p.parent.parent / "test/_annotations.coco.json").is_file()
    })
    if len(candidatas) != 1:
        raise RuntimeError(f"Especifica DATOS manualmente. Candidatas: {candidatas}")
    DATOS = candidatas[0]
DATOS = Path(DATOS)

coco = {}
for carpeta, particion in [("train", "train"), ("valid", "val"), ("test", "test")]:
    coco[particion] = json.loads((DATOS / carpeta / "_annotations.coco.json").read_text())
    contenido = coco[particion]
    print(particion, "| imágenes:", len(contenido["images"]),
          "| cajas:", len(contenido["annotations"]),
          "| categorías:", [(c["id"], c["name"]) for c in contenido["categories"]])

### Leer identidades, BCS y cajas

Las etiquetas `user_tags` contienen la identidad del animal y un valor como `BCS_2_75`, que convertimos a 2.75. Si una fotografía tiene identidades o etiquetas BCS ambiguas, conserva sus cajas para detección pero queda excluida de clasificación ordinal. No inferimos etiquetas a partir del nombre del archivo.

Comprobamos dimensiones y cajas, y construimos una tabla con una fila por foto. La huella SHA-256 permite detectar archivos idénticos entre particiones. La lectura de imágenes puede tardar un poco.

In [ ]:
# Esta función solo calcula la huella de un archivo, sin cargarlo entero en memoria.
def huella_archivo(ruta):
    huella = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            huella.update(bloque)
    return huella.hexdigest()

In [ ]:
filas = []
correcciones = []
for carpeta, particion in [("train", "train"), ("valid", "val"), ("test", "test")]:
    contenido = coco[particion]
    categorias = {c["id"]: c["name"] for c in contenido["categories"]}
    imagenes = {im["id"]: im for im in contenido["images"]}
    assert len(categorias) == len(contenido["categories"]), "Categorías duplicadas."
    assert {"cow", "ass"} <= set(categorias.values()), "Faltan clases cow/ass."
    assert len(imagenes) == len(contenido["images"]), "IDs de imagen duplicados."
    assert len({a["id"] for a in contenido["annotations"]}) == len(contenido["annotations"])
    anotaciones = defaultdict(list)
    for anotacion in contenido["annotations"]:
        assert anotacion["image_id"] in imagenes, "Referencia de imagen inválida."
        assert categorias[anotacion["category_id"]] in ("cow", "ass")
        assert anotacion.get("iscrowd", 0) == 0, "Revisar anotación iscrowd."
        anotaciones[anotacion["image_id"]].append(anotacion)

    nombres = set()
    for imagen in contenido["images"]:
        nombre = imagen["file_name"]
        assert Path(nombre).name == nombre and "\\" not in nombre
        assert nombre not in nombres, "Nombre de imagen duplicado."
        nombres.add(nombre)
        ruta = DATOS / carpeta / nombre
        with Image.open(ruta) as foto:
            assert foto.size == (imagen["width"], imagen["height"]), str(ruta)
            foto.verify()
        with Image.open(ruta) as foto:
            foto.convert("RGB").load()

        etiquetas = imagen.get("extra", {}).get("user_tags", [])
        identidades = sorted({str(t) for t in etiquetas if re.fullmatch(r"\d+", str(t))})
        etiquetas_bcs = sorted({str(t) for t in etiquetas if str(t).startswith("BCS_")})
        assert identidades, f"Falta identidad para comprobar la partición: {nombre}"
        bcs = None
        if len(identidades) == len(etiquetas_bcs) == 1:
            coincidencia = re.fullmatch(r"BCS_(\d+)(?:_(\d+))?", etiquetas_bcs[0])
            if coincidencia:
                bcs = float(coincidencia[1] + ("." + coincidencia[2] if coincidencia[2] else ""))
                assert 1 <= bcs <= 5, f"BCS fuera de rango: {nombre}"

        cajas = []
        for anotacion in anotaciones[imagen["id"]]:
            x, y, ancho, alto = anotacion["bbox"]
            assert all(math.isfinite(v) for v in (x, y, ancho, alto))
            assert ancho > 0 and alto > 0
            exceso = max(0, -x, -y, x + ancho - imagen["width"], y + alto - imagen["height"])
            assert exceso <= TOLERANCIA_CAJA + 1e-9, f"Caja fuera de imagen: {nombre}"
            xyxy = [max(0, x), max(0, y), min(imagen["width"], x + ancho),
                    min(imagen["height"], y + alto)]
            assert xyxy[2] > xyxy[0] and xyxy[3] > xyxy[1]
            if exceso > 1e-9:
                correcciones.append({"imagen": f"{carpeta}/{nombre}",
                                     "anotacion": anotacion["id"], "xyxy": xyxy})
            cajas.append({"clase": {"cow": 0, "ass": 1}[categorias[anotacion["category_id"]]],
                          "xyxy": xyxy})
        assert Counter(c["clase"] for c in cajas) == {0: 1, 1: 1}, nombre
        filas.append({"clave": f"{carpeta}/{nombre}", "ruta": str(ruta),
                      "particion": particion, "identidades": identidades,
                      "vaca": identidades[0] if len(identidades) == 1 else None,
                      "bcs": bcs, "sha256": huella_archivo(ruta),
                      "ancho": imagen["width"], "alto": imagen["height"], "cajas": cajas})

datos = pd.DataFrame(filas).sort_values("clave").reset_index(drop=True)
print("Fotos leídas:", len(datos), "| BCS válido:", datos["bcs"].notna().sum())
display(datos[["clave", "particion", "vaca", "bcs"]].head())

## 3. Comprobar la separación por animal

Una vaca no debe aparecer a la vez en entrenamiento y validación o test: de lo contrario podríamos medir reconocimiento del animal en lugar de capacidad de generalización. Verificamos **todas** las identidades, incluso las de fotos excluidas de clasificación ordinal, y buscamos archivos idénticos entre particiones.

Conservamos `train/valid/test` originales si pasan estas comprobaciones. Las huellas no detectan todas las fotos visualmente parecidas ni errores en las identidades; conviene revisar las anotaciones. Los valores de test no se usarán para elegir hiperparámetros.

In [ ]:
particiones_por_vaca = defaultdict(set)
particiones_por_huella = defaultdict(set)
for fila in filas:
    for identidad in fila["identidades"]:
        particiones_por_vaca[identidad].add(fila["particion"])
    particiones_por_huella[fila["sha256"]].add(fila["particion"])

vacas_repetidas = {v: sorted(p) for v, p in particiones_por_vaca.items() if len(p) > 1}
fotos_repetidas = {h: sorted(p) for h, p in particiones_por_huella.items() if len(p) > 1}
assert not vacas_repetidas, f"Vacas en varias particiones: {vacas_repetidas}"
assert not fotos_repetidas, "Existen archivos idénticos entre particiones."
for particion in ("train", "val", "test"):
    assert datos.loc[datos.particion == particion, "bcs"].notna().any(), particion

resumen = datos.groupby("particion").agg(
    fotos_detector=("clave", "size"), fotos_clasificador=("bcs", "count"), vacas=("vaca", "nunique")
)
resumen["excluidas_regresion"] = resumen.fotos_detector - resumen.fotos_clasificador
display(resumen)
datos.to_json(SALIDA / "datos.json", orient="records", indent=2)
(SALIDA / "correcciones_cajas.json").write_text(json.dumps(correcciones, indent=2))
datos.loc[datos.bcs.isna(), ["clave", "particion"]].to_csv(SALIDA / "excluidas_bcs.csv", index=False)

# Derivar el catálogo únicamente de las fotos de entrenamiento elegibles.
CLASES_BCS = sorted(float(v) for v in datos.loc[datos.particion == "train", "bcs"].dropna().unique())
assert len(CLASES_BCS) >= 2
assert all(abs((b - a) - 0.25) < 1e-6 for a, b in zip(CLASES_BCS, CLASES_BCS[1:])), "Revisar catálogo de entrenamiento: se esperan pasos de 0.25."
NUM_UMBRALES = len(CLASES_BCS) - 1
print("Clases ordinales derivadas de train:", CLASES_BCS)
display(datos[datos.particion == "train"].groupby("bcs").size().rename("fotos_train"))
(SALIDA / "clases_bcs.json").write_text(json.dumps(CLASES_BCS))

### Guardar la configuración del experimento

Guardamos parámetros y huellas de los datos para evitar reutilizar accidentalmente checkpoints con otro conjunto o configuración. Si cambias algo, crea otro `EXPERIMENTO`. Las rutas de entrada pueden cambiar al volver a adjuntar el mismo dataset.

In [ ]:
configuracion = {
    "version": "secuencial_4_ordinal_regularizado", "semilla": SEMILLA,
    "clases_bcs": CLASES_BCS, "metodo": "score_compartido_umbrales_ordenados",
    "decodificacion": "mediana_ordinal_logit_mayor_0", "test_previamente_observado": True,
    "datos": [{k: f[k] for k in ("clave", "particion", "sha256", "bcs", "cajas", "identidades")}
              for f in sorted(filas, key=lambda f: f["clave"])],
    "yolo": {"pesos": PESOS_YOLO, "epocas": EPOCAS_YOLO, "tam": TAM_YOLO,
             "lote": LOTE_YOLO, "paciencia": PACIENCIA_YOLO, "version": ULTRALYTICS_VERSION,
             "detector_previo_sha256": (huella_archivo(Path(DETECTOR_PREVIO) / "weights/best.pt")
                                        if DETECTOR_PREVIO is not None else None)},
    "clasificador": {"pesos": PESOS_EFFICIENTNET, "tam": TAM_RECORTE, "lote": LOTE_ORDINAL,
                 "epocas_cabeza": EPOCAS_CABEZA, "epocas_ajuste": EPOCAS_AJUSTE,
                 "lr_cabeza": LR_CABEZA, "lr_ajuste": LR_AJUSTE,
                 "capas_ajuste": CAPAS_AJUSTE, "early_stopping": False, "dropout": DROPOUT,
                 "optimizador": "AdamW", "weight_decay": WEIGHT_DECAY,
                 "excluir_decay": ["bias", "beta", "gamma", "inicio", "incrementos"],
                 "lr_factor": FACTOR_REDUCCION_LR, "lr_paciencia": PACIENCIA_LR,
                 "lr_minima": LR_MINIMA},
    "inferencia": {"confianza": CONFIANZA, "cobertura": COBERTURA_GRUPA}
}
ruta_configuracion = SALIDA / "configuracion.json"
if ruta_configuracion.exists():
    assert json.loads(ruta_configuracion.read_text()) == configuracion, "Usa otro EXPERIMENTO."
else:
    ruta_configuracion.write_text(json.dumps(configuracion, indent=2, allow_nan=False))

### Visualizar ejemplos de entrenamiento

Antes de entrenar, comprobamos que las cajas realmente rodean la vaca y su grupa. Aquí mostramos solo entrenamiento. El BCS es la etiqueta que aprenderá el clasificador; la identidad nunca será una entrada del modelo.

In [ ]:
ejemplos = datos[datos.particion == "train"].head(4)
fig, ejes = plt.subplots(1, len(ejemplos), figsize=(16, 5), squeeze=False)
for eje, (_, fila) in zip(ejes[0], ejemplos.iterrows()):
    with Image.open(fila.ruta) as foto:
        eje.imshow(foto.convert("RGB"))
    for caja in fila.cajas:
        x1, y1, x2, y2 = caja["xyxy"]
        color = "lime" if caja["clase"] == 0 else "orange"
        eje.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))
        eje.text(x1, y1, ["cow", "ass"][caja["clase"]], color=color)
    eje.set_title(f"BCS: {fila.bcs}")
    eje.axis("off")
plt.tight_layout()
plt.show()

## 4. Preparar las etiquetas para YOLO

YOLO espera una línea por objeto: `clase centro_x centro_y ancho alto`. Las coordenadas se normalizan por el tamaño de la imagen. Utilizamos **0 = vaca** y **1 = grupa**; no confundimos este mapeo con los IDs originales de COCO.

Las etiquetas y las copias de las fotos se guardan en `/kaggle/working`. La carpeta de entrada permanece de solo lectura. Esta exportación ocupa espacio, pero puede trasladarse a otra sesión.

In [ ]:
CARPETA_YOLO = SALIDA / "yolo_datos"
for fila in filas:
    nombre = fila["sha256"][:16] + "_" + Path(fila["ruta"]).name
    destino = CARPETA_YOLO / "images" / fila["particion"] / nombre
    etiqueta = CARPETA_YOLO / "labels" / fila["particion"] / (Path(nombre).stem + ".txt")
    destino.parent.mkdir(parents=True, exist_ok=True)
    etiqueta.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fila["ruta"], destino)
    lineas = []
    for caja in fila["cajas"]:
        x1, y1, x2, y2 = caja["xyxy"]
        cx = (x1 + x2) / (2 * fila["ancho"])
        cy = (y1 + y2) / (2 * fila["alto"])
        ancho = (x2 - x1) / fila["ancho"]
        alto = (y2 - y1) / fila["alto"]
        lineas.append(f"{caja['clase']} {cx:.10f} {cy:.10f} {ancho:.10f} {alto:.10f}")
    etiqueta.write_text("\n".join(lineas) + "\n")

RUTA_YAML = SALIDA / "detector.yaml"
# JSON es compatible con YAML y evita problemas al escribir rutas.
RUTA_YAML.write_text(json.dumps({
    "path": str(CARPETA_YOLO), "train": "images/train", "val": "images/val",
    "test": "images/test", "names": {0: "cow", 1: "ass"}
}, indent=2))
print(RUTA_YAML.read_text())

## 5. Entrenar YOLO

Por defecto se entrena YOLO durante 80 épocas desde `yolo26n.pt`, con paciencia 0 (sin parada temprana). No se necesitan pesos del experimento anterior. Si esta misma ejecución ya tiene checkpoints en working, el flujo conserva la posibilidad de reanudar o reutilizar una fase finalizada.

Conservamos confianza 0.25 y la regla geométrica de asociación. Las posibles abstenciones no se resuelven cambiando la cabeza BCS. El clasificador ordinal se entrenará después desde ImageNet; no se cargan pesos de regresión.


In [ ]:
CARPETA_DETECTOR = SALIDA / "detector"
ULTIMO_YOLO = CARPETA_DETECTOR / "weights/last.pt"
MEJOR_YOLO = CARPETA_DETECTOR / "weights/best.pt"

if DETECTOR_PREVIO is not None:
    origen_detector = Path(DETECTOR_PREVIO)
    assert (origen_detector / "weights/best.pt").is_file(), "Falta best.pt del detector previo."
    assert (origen_detector / "results.csv").is_file(), "Falta results.csv del detector previo."
    if not CARPETA_DETECTOR.exists():
        shutil.copytree(origen_detector, CARPETA_DETECTOR)
    assert huella_archivo(MEJOR_YOLO) == huella_archivo(origen_detector / "weights/best.pt"), "Detector previo diferente."
    detector = YOLO(str(MEJOR_YOLO))
    print("Reutilizando el detector previo; este experimento cambia solo la clasificación ordinal.")
elif ULTIMO_YOLO.exists():
    detector = YOLO(str(ULTIMO_YOLO))
    if detector.ckpt.get("optimizer") is not None:
        detector.train(resume=True, device=0, patience=0)
    else:
        assert MEJOR_YOLO.exists(), "Falta best.pt; revisa los outputs del entrenamiento."
        print("Detector ya finalizado; se utilizará best.pt.")
else:
    detector = YOLO(PESOS_YOLO)
    detector.train(
        data=str(RUTA_YAML), epochs=EPOCAS_YOLO, imgsz=TAM_YOLO,
        batch=LOTE_YOLO, device=0, patience=PACIENCIA_YOLO,
        seed=SEMILLA, workers=2, project=str(SALIDA), name="detector",
        exist_ok=True, save=True, save_period=5
    )
assert MEJOR_YOLO.exists(), "No se encontró el mejor checkpoint del detector."

### Revisar las curvas del detector

Las pérdidas describen el ajuste y mAP resume la calidad de localización y clasificación de las cajas. Una mejora en detección no garantiza una mejora del BCS: más adelante mediremos también el pipeline completo.

In [ ]:
historial_yolo = pd.read_csv(CARPETA_DETECTOR / "results.csv")
historial_yolo.columns = historial_yolo.columns.str.strip()
display(historial_yolo.tail())
curvas = [c for c in historial_yolo if "mAP" in c or c.endswith("box_loss")]
historial_yolo.plot(x="epoch", y=curvas, subplots=True, figsize=(10, 9), grid=True)
plt.tight_layout()
plt.show()

# Quitamos referencias a YOLO antes de inicializar TensorFlow.
del detector
gc.collect()
torch.cuda.empty_cache()

## 6. Construir los recortes de grupa

El clasificador aprende con las **cajas anotadas** de entrenamiento. Si utilizáramos las predicciones del detector para etiquetar o seleccionar estas muestras, mezclaríamos errores de dos tareas durante el aprendizaje.

Recortamos la grupa, conservamos su proporción y añadimos relleno negro hasta llegar a 224 × 224. Esta misma función se usará con cajas predichas. Redondeamos hacia fuera para incluir toda la caja.

In [ ]:
def recortar_grupa(imagen, caja):
    x1, y1, x2, y2 = caja
    limites = (max(0, math.floor(x1)), max(0, math.floor(y1)),
               min(imagen.width, math.ceil(x2)), min(imagen.height, math.ceil(y2)))
    assert limites[2] > limites[0] and limites[3] > limites[1], "Recorte vacío."
    region = imagen.convert("RGB").crop(limites)
    return ImageOps.pad(region, (TAM_RECORTE, TAM_RECORTE),
                        method=Image.Resampling.BILINEAR, color=(0, 0, 0))

In [ ]:
muestras = []
for fila in filas:
    if fila["bcs"] is None:
        continue
    caja_grupa = next(c["xyxy"] for c in fila["cajas"] if c["clase"] == 1)
    nombre = hashlib.sha256(fila["clave"].encode()).hexdigest() + ".png"
    destino = SALIDA / "recortes_gt" / fila["particion"] / nombre
    destino.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(fila["ruta"]) as foto:
        recortar_grupa(foto, caja_grupa).save(destino)
    muestras.append({"clave": fila["clave"], "particion": fila["particion"],
                     "bcs": fila["bcs"], "recorte": str(destino)})
muestras = pd.DataFrame(muestras)
muestras.to_csv(SALIDA / "muestras_regresion.csv", index=False)
display(muestras.groupby("particion").agg(fotos=("clave", "size")))

In [ ]:
ejemplos = muestras[muestras.particion == "train"].head(6)
fig, ejes = plt.subplots(1, len(ejemplos), figsize=(15, 3), squeeze=False)
for eje, (_, fila) in zip(ejes[0], ejemplos.iterrows()):
    with Image.open(fila.recorte) as imagen:
        eje.imshow(imagen)
    eje.set_title(f"BCS: {fila.bcs:.2f}")
    eje.axis("off")
plt.tight_layout()
plt.show()

## 7. Preparar etiquetas ordinales y lotes

Para K clases ordenadas se crean K−1 etiquetas binarias: `BCS > CLASES_BCS[k]`. No son etiquetas one-hot. Solo entrenamiento se baraja; todas las fotos de una vaca permanecen en su partición original.

Se mantiene RGB en 0–255 porque EfficientNet integra su escalado. Un BCS de validación fuera del catálogo no se aproxima silenciosamente a otra clase.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

gpus = tf.config.list_physical_devices("GPU")
assert gpus, "TensorFlow no detecta GPU."
try:
    tf.config.set_visible_devices(gpus[0], "GPU")
    tf.config.experimental.set_memory_growth(gpus[0], True)
except RuntimeError:
    print("GPU ya inicializada; si necesitas cambiar su configuración, reinicia el kernel.")
keras.utils.set_random_seed(SEMILLA)
print("TensorFlow:", tf.__version__)

In [ ]:
def leer_recorte(ruta, bcs):
    imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
    imagen = tf.ensure_shape(imagen, (TAM_RECORTE, TAM_RECORTE, 3))
    objetivos = tf.cast(bcs > tf.constant(CLASES_BCS[:-1], dtype=tf.float32), tf.float32)
    return tf.cast(imagen, tf.float32), objetivos

conjuntos = {}
for particion in ("train", "val"):
    seleccion = muestras[muestras.particion == particion]
    assert seleccion.bcs.isin(CLASES_BCS).all(), "Hay BCS fuera del catálogo de entrenamiento."
    conjunto = tf.data.Dataset.from_tensor_slices((
        seleccion.recorte.to_numpy(dtype=str), seleccion.bcs.to_numpy(dtype=np.float32)
    ))
    if particion == "train":
        conjunto = conjunto.shuffle(len(seleccion), seed=SEMILLA)
    conjuntos[particion] = conjunto.map(leer_recorte, num_parallel_calls=tf.data.AUTOTUNE).batch(LOTE_ORDINAL).prefetch(tf.data.AUTOTUNE)

imagenes_lote, objetivos_lote = next(iter(conjuntos["train"]))
print("Imágenes:", imagenes_lote.shape, "| Objetivos ordinales:", objetivos_lote.shape)
print(objetivos_lote[:4].numpy())

## 8. Cabeza ordinal con umbrales ordenados

La base produce características y una neurona genera un score s. Para cada límite se obtiene un logit `s − t[k]`. Los umbrales t se construyen con incrementos positivos (`softplus`), de modo que las probabilidades de superar categorías más altas nunca pueden aumentar.

La pérdida es BinaryCrossentropy(from_logits=True), promediada por umbral. La métrica `mae` convierte las etiquetas binarias y los logits a categorías BCS y calcula el error en unidades BCS. **MAE solo selecciona y evalúa; no es la pérdida de entrenamiento.**

Las clases personalizadas se registran para guardar y cargar modelos completos. En una sesión nueva hay que ejecutar estas definiciones antes de `load_model`. El checkpoint ordinal no es compatible con el notebook de test de regresión v2.

In [ ]:
@keras.utils.register_keras_serializable(package="BCSOrdinal")
class UmbralesOrdenados(layers.Layer):
    def __init__(self, num_umbrales, **kwargs):
        super().__init__(**kwargs)
        self.num_umbrales = int(num_umbrales)

    def build(self, input_shape):
        self.inicio = self.add_weight(name="inicio", shape=(), initializer=keras.initializers.Constant(-1.5))
        if self.num_umbrales > 1:
            self.incrementos = self.add_weight(name="incrementos", shape=(self.num_umbrales - 1,), initializer="zeros")
        super().build(input_shape)

    def call(self, score):
        primero = tf.reshape(self.inicio, (1,))
        if self.num_umbrales > 1:
            cortes = tf.concat([primero, self.inicio + tf.cumsum(tf.nn.softplus(self.incrementos) + 1e-4)], axis=0)
        else:
            cortes = primero
        return score - cortes

    def get_config(self):
        return {**super().get_config(), "num_umbrales": self.num_umbrales}


@keras.utils.register_keras_serializable(package="BCSOrdinal")
class MAEOrdinal(keras.metrics.Mean):
    def __init__(self, clases, name="mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.clases = list(clases)

    def update_state(self, y_true, y_pred, sample_weight=None):
        clases = tf.constant(self.clases, dtype=tf.float32)
        real = tf.gather(clases, tf.cast(tf.reduce_sum(y_true, axis=-1), tf.int32))
        rango = tf.reduce_sum(tf.cast(y_pred > 0, tf.int32), axis=-1)
        predicho = tf.gather(clases, rango)
        return super().update_state(tf.abs(predicho - real), sample_weight=sample_weight)

    def get_config(self):
        return {**super().get_config(), "clases": self.clases}


def decodificar_ordinal(logits):
    q = tf.math.sigmoid(tf.convert_to_tensor(logits)).numpy()
    assert np.isfinite(q).all()
    assert np.all(np.diff(q, axis=-1) <= 1e-6), "Probabilidades ordinales inconsistentes."
    probabilidades = np.concatenate([1 - q[:, :1], q[:, :-1] - q[:, 1:], q[:, -1:]], axis=1)
    assert np.all(probabilidades >= -1e-6) and np.allclose(probabilidades.sum(axis=1), 1, atol=1e-6)
    categorias = np.asarray(CLASES_BCS)[np.sum(np.asarray(logits) > 0, axis=-1)]
    esperado = probabilidades @ np.asarray(CLASES_BCS)
    return categorias, esperado, probabilidades


@keras.utils.register_keras_serializable(package="BCSOrdinal")
class AdamWOrdinal(keras.optimizers.AdamW):
    def build(self, var_list):
        if not self.built:
            self.exclude_from_weight_decay(var_names=["bias", "beta", "gamma", "inicio", "incrementos"])
        super().build(var_list)


def crear_optimizador(lr):
    # La subclase restaura también las exclusiones al cargar el optimizador.
    return AdamWOrdinal(learning_rate=lr, weight_decay=WEIGHT_DECAY)


entrada = keras.Input(shape=(TAM_RECORTE, TAM_RECORTE, 3), name="rgb_0_255")
aumentos = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEMILLA),
    layers.RandomRotation(0.03, seed=SEMILLA + 1),
    layers.RandomZoom(0.05, seed=SEMILLA + 2)
], name="aumentos_entrenamiento")
base = keras.applications.EfficientNetB0(
    include_top=False, weights=PESOS_EFFICIENTNET,
    input_shape=(TAM_RECORTE, TAM_RECORTE, 3)
)
base.trainable = False
x = base(aumentos(entrada), training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT, seed=SEMILLA + 3)(x)
score = layers.Dense(1, use_bias=False, dtype="float32", name="score_ordinal")(x)
salida_bcs = UmbralesOrdenados(NUM_UMBRALES, name="umbrales_ordenados")(score)
clasificador = keras.Model(entrada, salida_bcs, name="clasificador_bcs")
clasificador.compile(optimizer=crear_optimizador(LR_CABEZA), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[MAEOrdinal(CLASES_BCS)])
clasificador.summary()

### Guardar el último modelo, el mejor y reducir la tasa de aprendizaje

`best.keras` conserva el mejor MAE de validación y `last.keras` el último modelo completo, incluido su optimizador. Un archivo CSV permite revisar las curvas y conocer las épocas finalizadas. Cada fase tiene su propia carpeta.

No utilizamos `EarlyStopping`: las dos fases recorren todas sus épocas. En su lugar, `ReduceLROnPlateau` divide la tasa de aprendizaje entre dos tras 8 épocas sin mejora de validación, hasta un mínimo de `1e-7`. Esto permite seguir entrenando con pasos más pequeños sin detener el proceso. Conservamos `TerminateOnNaN` para interrumpir un entrenamiento numéricamente inválido.

Si vuelves a ejecutar después de una interrupción, retomamos el último modelo y el número de épocas registrado. El optimizador conserva la tasa guardada, pero el contador de estancamiento del callback comienza de nuevo. Esta versión no recupera transaccionalmente todos los estados; si la sesión termina durante el guardado, comprueba los archivos antes de continuar.

Esta pequeña función reúne los mismos callbacks para las dos fases. Las llamadas a `fit` permanecen explícitas en sus celdas. El reductor se ejecuta antes del guardado para conservar la tasa actualizada en el checkpoint.


In [ ]:
def callbacks_fase(carpeta):
    mejor_previo = None
    if (carpeta / "history.csv").exists():
        mejor_previo = float(pd.read_csv(carpeta / "history.csv").val_mae.min())
    return [
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae", mode="min", factor=FACTOR_REDUCCION_LR,
            patience=PACIENCIA_LR, min_delta=1e-4, min_lr=LR_MINIMA, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(str(carpeta / "best.keras"), monitor="val_mae",
                                        mode="min", save_best_only=True,
                                        initial_value_threshold=mejor_previo),
        keras.callbacks.ModelCheckpoint(str(carpeta / "last.keras")),
        keras.callbacks.CSVLogger(str(carpeta / "history.csv"), append=True),
        keras.callbacks.TerminateOnNaN()
    ]


## 9. Entrenar la cabeza de clasificación ordinal

Validación se utiliza durante `fit` para seleccionar el mejor modelo. Test no participa. Si existe `completa.json`, esta fase ya terminó; si solo existe el último checkpoint, continuamos hasta el objetivo total de épocas.

In [ ]:
CABEZA = SALIDA / "clasificador/cabeza"
CABEZA.mkdir(parents=True, exist_ok=True)
if not (CABEZA / "completa.json").exists():
    epoca_inicial = 0
    if (CABEZA / "last.keras").exists():
        clasificador = keras.models.load_model(CABEZA / "last.keras")
        historial = pd.read_csv(CABEZA / "history.csv")
        epoca_inicial = int(historial.epoch.max()) + 1
    if epoca_inicial < EPOCAS_CABEZA:
        clasificador.fit(
            conjuntos["train"], validation_data=conjuntos["val"],
            initial_epoch=epoca_inicial, epochs=EPOCAS_CABEZA,
            callbacks=callbacks_fase(CABEZA)
        )
    assert (CABEZA / "best.keras").exists()
    registro = pd.read_csv(CABEZA / "history.csv")
    assert np.isfinite(registro[["loss", "val_loss", "mae", "val_mae"]].to_numpy()).all(), "Pérdida no finita: revisa el entrenamiento."
    (CABEZA / "completa.json").write_text(json.dumps({"estado": "completa"}))
else:
    print("Fase de cabeza ya completada.")

## 10. Ajustar las últimas capas

Ahora cargamos la mejor cabeza y permitimos modificar las últimas 20 capas de EfficientNet, manteniendo BatchNormalization congelada. Reducimos la tasa de aprendizaje de `3e-4` a `5e-6` para ajustar gradualmente las características preentrenadas.

Al cambiar qué capas son entrenables hay que recompilar. Al **reanudar** un ajuste ya iniciado cargamos el modelo completo sin recompilar, para conservar su optimizador.

In [ ]:
AJUSTE = SALIDA / "clasificador/ajuste_fino"
AJUSTE.mkdir(parents=True, exist_ok=True)
if not (AJUSTE / "completa.json").exists():
    epoca_inicial = 0
    if (AJUSTE / "last.keras").exists():
        clasificador = keras.models.load_model(AJUSTE / "last.keras")
        historial = pd.read_csv(AJUSTE / "history.csv")
        epoca_inicial = int(historial.epoch.max()) + 1
    else:
        clasificador = keras.models.load_model(CABEZA / "best.keras")
        base = clasificador.get_layer("efficientnetb0")
        base.trainable = True
        for indice, capa in enumerate(base.layers):
            capa.trainable = (indice >= len(base.layers) - CAPAS_AJUSTE
                              and not isinstance(capa, layers.BatchNormalization))
        clasificador.compile(optimizer=crear_optimizador(LR_AJUSTE), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[MAEOrdinal(CLASES_BCS)])
    if epoca_inicial < EPOCAS_AJUSTE:
        clasificador.fit(
            conjuntos["train"], validation_data=conjuntos["val"],
            initial_epoch=epoca_inicial, epochs=EPOCAS_AJUSTE,
            callbacks=callbacks_fase(AJUSTE)
        )
    assert (AJUSTE / "best.keras").exists()
    registro = pd.read_csv(AJUSTE / "history.csv")
    assert np.isfinite(registro[["loss", "val_loss", "mae", "val_mae"]].to_numpy()).all(), "Pérdida no finita: revisa el entrenamiento."
    (AJUSTE / "completa.json").write_text(json.dumps({"estado": "completa"}))
else:
    print("Ajuste fino ya completado.")

### Comparar las fases usando validación

El ajuste fino puede empeorar el resultado. Por eso elegimos entre las mejores versiones de ambas fases utilizando **solo MAE de validación**, y guardamos la selección para la evaluación posterior.

In [ ]:
historiales = {}
for nombre, carpeta in [("cabeza", CABEZA), ("ajuste_fino", AJUSTE)]:
    historiales[nombre] = pd.read_csv(carpeta / "history.csv")
    historiales[nombre].plot(x="epoch", y=["mae", "val_mae"], title=nombre, grid=True)
    plt.ylabel("Error absoluto medio (BCS)")
    plt.show()

mejores_mae = {nombre: float(h.val_mae.min()) for nombre, h in historiales.items()}
fase_elegida = min(mejores_mae, key=mejores_mae.get)
MEJOR_ORDINAL = (CABEZA if fase_elegida == "cabeza" else AJUSTE) / "best.keras"
clasificador = keras.models.load_model(MEJOR_ORDINAL)
print("MAE de validación:", mejores_mae)
print("Fase seleccionada:", fase_elegida)
(SALIDA / "seleccion_clasificador.json").write_text(json.dumps({
    "fase": fase_elegida, "mae_validacion": mejores_mae,
    "checkpoint": str(MEJOR_ORDINAL.relative_to(SALIDA))
}, indent=2))

comparacion = pd.DataFrame([
    {"modelo": "ordinal_v3_validacion", "mae": 0.2638888955116272},
    {"modelo": "regresion_v2_validacion", "mae": 0.2346049100160598},
    {"modelo": EXPERIMENTO, "mae": mejores_mae[fase_elegida]}
])
display(comparacion)
comparacion.to_csv(SALIDA / "comparacion_validacion.csv", index=False)

# Medir la brecha sin dropout ni aumentos activos, usando el checkpoint elegido.
evaluaciones_deterministas = []
for particion in ("train", "val"):
    seleccion = muestras[muestras.particion == particion]
    ds = tf.data.Dataset.from_tensor_slices((seleccion.recorte.to_numpy(dtype=str),
                                            seleccion.bcs.to_numpy(dtype=np.float32)))
    ds = ds.map(leer_recorte).batch(LOTE_ORDINAL)
    valores = clasificador.evaluate(ds, return_dict=True, verbose=0)
    evaluaciones_deterministas.append({"particion": particion, **{k: float(v) for k, v in valores.items()}})
display(pd.DataFrame(evaluaciones_deterministas))
(SALIDA / "brecha_determinista.json").write_text(json.dumps(evaluaciones_deterministas, indent=2))

## 11. Preparar la evaluación del pipeline

Mediremos tres situaciones sobre las mismas fotos con BCS válido:

- **Base de comparación:** predecir siempre la mediana del BCS de entrenamiento.
- **Clasificador con caja anotada:** mide el BCS suponiendo una localización correcta.
- **Pipeline completo:** YOLO localiza y EfficientNet estima; puede abstenerse si no hay una asociación única entre vaca y grupa.

La grupa debe tener su centro dentro de la vaca y al menos el 80 % de su área contenida en ella. Si hay varios pares válidos no elegimos arbitrariamente uno. La confianza de YOLO describe la detección, no la confianza del BCS.

In [ ]:
def asociar_grupa(resultado):
    cajas = []
    for coordenadas, clase in zip(resultado.boxes.xyxy.cpu().tolist(),
                                  resultado.boxes.cls.cpu().tolist()):
        cajas.append({"clase": int(clase), "xyxy": coordenadas})
    pares = []
    for vaca in [c for c in cajas if c["clase"] == 0]:
        a, b, c, d = vaca["xyxy"]
        for grupa in [g for g in cajas if g["clase"] == 1]:
            x1, y1, x2, y2 = grupa["xyxy"]
            area = (x2-x1) * (y2-y1)
            interseccion = max(0, min(c, x2)-max(a, x1)) * max(0, min(d, y2)-max(b, y1))
            centro_dentro = a <= (x1+x2)/2 <= c and b <= (y1+y2)/2 <= d
            if area > 0 and centro_dentro and interseccion/area >= COBERTURA_GRUPA:
                pares.append(grupa["xyxy"])
    if len(pares) == 1:
        return pares[0], "ok"
    return None, "sin_par_valido" if not pares else "asociacion_ambigua"

### Evaluación exploratoria y selección fijada

Por defecto evaluamos validación. El conjunto test de las versiones anteriores ya fue observado. Si posteriormente se ejecuta `PARTICION_EVALUACION="test"`, el resultado se identifica como exploratorio; no sustituye una evaluación final con nuevos animales.

Fija antes de ejecutarlo la fase, el método ordinal y los umbrales. No cambies el criterio de decodificación según test.

In [ ]:
assert PARTICION_EVALUACION in ("val", "test")
seleccion_final = {
    "detector": huella_archivo(MEJOR_YOLO),
    "clasificador": huella_archivo(MEJOR_ORDINAL),
    "configuracion": huella_archivo(ruta_configuracion),
    "metodo": "ordinal_mediana", "test_previamente_observado": True
}
if PARTICION_EVALUACION == "test":
    assert TEST_FINAL_CONFIRMADO, "Cierra decisiones en validación antes de abrir test."
    bloqueo = SALIDA / "seleccion_test.json"
    if bloqueo.exists():
        assert json.loads(bloqueo.read_text()) == seleccion_final, "La selección cambió después de abrir test."
    else:
        bloqueo.write_text(json.dumps(seleccion_final, indent=2))

### Evaluar la detección

Precisión, recall y mAP describen las cajas en todas las imágenes de la partición. A continuación liberamos YOLO antes de aplicar el clasificador a los recortes, reduciendo su coincidencia en memoria.

In [ ]:
# Liberamos el modelo de Keras antes de volver a cargar YOLO.
del clasificador, base, entrada, x, salida_bcs, aumentos, score
keras.backend.clear_session()
gc.collect()
detector = YOLO(str(MEJOR_YOLO))
metricas_detector = detector.val(
    data=str(RUTA_YAML), split=PARTICION_EVALUACION,
    imgsz=TAM_YOLO, batch=LOTE_YOLO, device=0,
    project=str(SALIDA), name="deteccion_" + PARTICION_EVALUACION, exist_ok=True
)
resultado_detector = {k: float(v) for k, v in metricas_detector.results_dict.items()}
display(pd.Series(resultado_detector, name="detección"))
(SALIDA / f"metricas_detector_{PARTICION_EVALUACION}.json").write_text(
    json.dumps(resultado_detector, indent=2)
)

### Detectar y guardar los recortes predichos

Aquí ejecutamos solamente YOLO. Guardamos la causa de abstención y, cuando hay un par válido, el recorte de la grupa. Después ejecutaremos EfficientNet sobre estos recortes usando exactamente el mismo preprocesamiento que con las cajas anotadas.

In [ ]:
evaluacion = datos[(datos.particion == PARTICION_EVALUACION) & datos.bcs.notna()].copy()
carpeta_predichos = SALIDA / "recortes_predichos" / PARTICION_EVALUACION
carpeta_predichos.mkdir(parents=True, exist_ok=True)
detecciones = []
for fila in evaluacion.itertuples():
    resultado = detector.predict(fila.ruta, conf=CONFIANZA, imgsz=TAM_YOLO, device=0, verbose=False)[0]
    caja, estado = asociar_grupa(resultado)
    ruta_recorte = None
    if caja is not None:
        nombre = hashlib.sha256(fila.clave.encode()).hexdigest() + ".png"
        ruta_recorte = str(carpeta_predichos / nombre)
        with Image.open(fila.ruta) as foto:
            recortar_grupa(foto, caja).save(ruta_recorte)
    detecciones.append({"clave": fila.clave, "estado": estado,
                         "caja_predicha": caja, "recorte_predicho": ruta_recorte})
    del resultado

del detector, metricas_detector
gc.collect()
torch.cuda.empty_cache()
print(Counter(d["estado"] for d in detecciones))

### Predecir categorías y medir BCS

La categoría principal se obtiene contando logits positivos (probabilidad de superar el umbral >0.5), que corresponde a una mediana de la distribución ordinal. El valor esperado se guarda como diagnóstico continuo y no reemplaza la salida principal según qué métrica resulte más favorable.

Reportamos MAE, RMSE, sesgo, R², tolerancias y exactitud de categoría. Se conserva cobertura del pipeline y la comparación con cajas anotadas sobre las mismas fotos.

In [ ]:
clasificador = keras.models.load_model(MEJOR_ORDINAL)
mediana_entrenamiento = float(muestras.loc[muestras.particion == "train", "bcs"].median())
rutas_gt = muestras.set_index("clave").recorte.to_dict()
assert evaluacion.bcs.isin(CLASES_BCS).all(), "Etiqueta evaluada fuera del catálogo de entrenamiento."
resultados_bcs = []
for fila, deteccion in zip(evaluacion.itertuples(), detecciones):
    with Image.open(rutas_gt[fila.clave]) as recorte:
        lote = np.asarray(recorte, dtype=np.float32)[None, ...]
    categoria_gt, esperado_gt, probs_gt = decodificar_ordinal(clasificador.predict(lote, verbose=0))
    pred_gt = float(categoria_gt[0])
    pred_pipeline = None
    esperado_pipeline = None
    probabilidades_pipeline = None
    if deteccion["recorte_predicho"] is not None:
        with Image.open(deteccion["recorte_predicho"]) as recorte:
            lote = np.asarray(recorte, dtype=np.float32)[None, ...]
        categoria, esperado, probs = decodificar_ordinal(clasificador.predict(lote, verbose=0))
        pred_pipeline = float(categoria[0])
        esperado_pipeline = float(esperado[0])
        probabilidades_pipeline = probs[0].tolist()
    resultados_bcs.append({**deteccion, "real": fila.bcs,
                           "base": mediana_entrenamiento,
                           "ordinal_gt": pred_gt, "pipeline": pred_pipeline,
                           "esperado_gt": float(esperado_gt[0]), "probabilidades_gt": probs_gt[0].tolist(),
                           "esperado_pipeline": esperado_pipeline, "probabilidades_pipeline": probabilidades_pipeline})
predicciones = pd.DataFrame(resultados_bcs)
predicciones.to_csv(SALIDA / f"predicciones_{PARTICION_EVALUACION}.csv", index=False)

In [ ]:
def medir_bcs(tabla, columna):
    disponibles = tabla[columna].notna()
    real = tabla.loc[disponibles, "real"].to_numpy(dtype=float)
    predicho = tabla.loc[disponibles, columna].to_numpy(dtype=float)
    assert np.isfinite(real).all() and np.isfinite(predicho).all()
    error = predicho - real
    total = len(tabla)
    resultado = {"evaluables": total, "predicciones": len(error),
                 "abstenciones": total - len(error),
                 "cobertura": len(error)/total if total else None,
                 "exactitud_categoria": None, "mae": None, "rmse": None, "sesgo": None, "r2": None,
                 "error_hasta_025": None, "error_hasta_050": None}
    if len(error):
        denominador = np.sum((real - real.mean())**2)
        resultado.update(exactitud_categoria=float(np.mean(np.isclose(predicho, real))), mae=float(np.abs(error).mean()), rmse=float(np.sqrt(np.mean(error**2))),
                         sesgo=float(error.mean()),
                         r2=float(1 - np.sum(error**2)/denominador) if denominador > 0 else None,
                         error_hasta_025=float(np.mean(np.abs(error) <= 0.25)),
                         error_hasta_050=float(np.mean(np.abs(error) <= 0.50)))
    return resultado

metricas = {columna: medir_bcs(predicciones, columna)
            for columna in ("base", "ordinal_gt", "pipeline")}
cubiertas = predicciones[predicciones.pipeline.notna()]
for columna in ("base", "ordinal_gt"):
    metricas[columna + "_mismas_fotos"] = medir_bcs(cubiertas, columna)
display(pd.DataFrame(metricas).T)
(SALIDA / f"metricas_bcs_{PARTICION_EVALUACION}.json").write_text(
    json.dumps(metricas, indent=2, allow_nan=False)
)

In [ ]:
# Matriz de confusión BCS para las categorías ordinales, sin imputar abstenciones.
for columna in ("ordinal_gt", "pipeline"):
    cubiertas_clase = predicciones.dropna(subset=[columna])
    matriz = pd.crosstab(cubiertas_clase.real, cubiertas_clase[columna]).reindex(index=CLASES_BCS, columns=CLASES_BCS, fill_value=0)
    display(matriz)
    matriz.to_csv(SALIDA / f"confusion_{columna}_{PARTICION_EVALUACION}.csv")
print("Orden de probabilidades en el CSV:", CLASES_BCS)


## 12. Interpretar los resultados

La diagonal real–predicho representa acierto. Ahora las predicciones principales pertenecen al catálogo ordinal. Comprueba MAE, exactitud, tolerancias y cobertura junto a la mediana de entrenamiento. La clasificación ordinal puede fallar y no debe declararse mejor antes de medirlo.

El scatter y las tablas son por foto. En las mismas fotos cubiertas, la diferencia entre caja anotada y predicha permite diagnosticar la sensibilidad al recorte. Las abstenciones siguen correspondiendo a la asociación de YOLO.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, columna in zip(ejes, ["ordinal_gt", "pipeline"]):
    seleccion = predicciones[predicciones[columna].notna()]
    eje.scatter(seleccion.real, seleccion[columna], alpha=0.65)
    eje.plot([1, 5], [1, 5], "k--")
    eje.set(xlabel="BCS real", ylabel="BCS predicho", title=columna)
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

predicciones["error_absoluto"] = (predicciones.pipeline - predicciones.real).abs()
peores = predicciones.dropna(subset=["pipeline"]).nlargest(4, "error_absoluto")
if len(peores):
    fig, ejes = plt.subplots(1, len(peores), figsize=(14, 4), squeeze=False)
    originales = datos.set_index("clave").ruta.to_dict()
    for eje, fila in zip(ejes[0], peores.itertuples()):
        with Image.open(originales[fila.clave]) as foto:
            eje.imshow(foto.convert("RGB"))
        x1, y1, x2, y2 = fila.caja_predicha
        eje.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="orange", linewidth=2))
        eje.set_title(f"Real {fila.real:.2f} | Pred. {fila.pipeline:.2f}")
        eje.axis("off")
    plt.tight_layout()
    plt.show()
display(predicciones.loc[predicciones.pipeline.isna(), ["clave", "estado"]].head(10))

## Tabla completa por observación

Mostramos **todas las fotos de la partición**, sin truncar la tabla. `bcs_predicho` y su error corresponden al pipeline completo. En una abstención estos valores quedan vacíos: no los sustituimos por cero ni por la predicción con caja anotada.

Las columnas `bcs_predicho_caja_anotada` y sus errores muestran separadamente el rendimiento del clasificador suponiendo una localización correcta. El error con signo es **predicho − real**: positivo indica sobreestimación; negativo, subestimación. El error absoluto mide la magnitud del fallo.

El CSV conserva toda la precisión; solo redondeamos la visualización a cuatro decimales. La columna `observacion` permite reportar los casos sin publicar identificadores de animales.

In [ ]:
tabla_observaciones = datos.loc[
    datos.particion == PARTICION_EVALUACION, ["clave", "bcs"]
].rename(columns={"bcs": "bcs_real"}).merge(
    predicciones[["clave", "ordinal_gt", "pipeline", "estado"]],
    on="clave", how="left", validate="one_to_one"
).rename(columns={"ordinal_gt": "bcs_predicho_caja_anotada", "pipeline": "bcs_predicho"})
tabla_observaciones["estado"] = tabla_observaciones.estado.fillna("bcs_real_no_evaluable")
tabla_observaciones["error"] = tabla_observaciones.bcs_predicho - tabla_observaciones.bcs_real
tabla_observaciones["error_absoluto"] = tabla_observaciones.error.abs()
tabla_observaciones["error_caja_anotada"] = (
    tabla_observaciones.bcs_predicho_caja_anotada - tabla_observaciones.bcs_real
)
tabla_observaciones["error_absoluto_caja_anotada"] = tabla_observaciones.error_caja_anotada.abs()
tabla_observaciones = tabla_observaciones.sort_values("clave").reset_index(drop=True)
tabla_observaciones.insert(0, "observacion", np.arange(1, len(tabla_observaciones) + 1))
assert len(tabla_observaciones) == int((datos.particion == PARTICION_EVALUACION).sum())
assert tabla_observaciones.clave.is_unique
ruta_tabla = SALIDA / f"tabla_observaciones_{PARTICION_EVALUACION}.csv"
tabla_observaciones.to_csv(ruta_tabla, index=False)
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.max_colwidth", None, "display.float_format", "{:.4f}".format):
    display(tabla_observaciones)
print("Tabla completa guardada en:", ruta_tabla)

## 13. Predecir una fotografía nueva (opcional)

Configura `IMAGEN_NUEVA` con una imagen adjunta y ejecuta esta celda. Primero detectamos la vaca y la grupa; después estimamos BCS con el recorte. Si no hay asociación única, mostramos una abstención en lugar de inventar un valor.

In [ ]:
if IMAGEN_NUEVA is not None:
    del clasificador
    keras.backend.clear_session()
    gc.collect()
    detector = YOLO(str(MEJOR_YOLO))
    resultado = detector.predict(str(IMAGEN_NUEVA), conf=CONFIANZA,
                                 imgsz=TAM_YOLO, device=0, verbose=False)[0]
    caja, estado = asociar_grupa(resultado)
    display(Image.fromarray(resultado.plot()[..., ::-1]))
    del detector, resultado
    gc.collect()
    torch.cuda.empty_cache()
    clasificador = keras.models.load_model(MEJOR_ORDINAL)
    prediccion_nueva = {"estado": estado, "bcs": None}
    if caja is not None:
        with Image.open(IMAGEN_NUEVA) as foto:
            recorte = recortar_grupa(foto, caja)
        display(recorte)
        lote = np.asarray(recorte, dtype=np.float32)[None, ...]
        categorias, esperado, probabilidades = decodificar_ordinal(clasificador.predict(lote, verbose=0))
        prediccion_nueva.update(bcs=float(categorias[0]), bcs_esperado=float(esperado[0]),
                                clases=CLASES_BCS, probabilidades=probabilidades[0].tolist())
    print(prediccion_nueva)
    (SALIDA / "prediccion_nueva.json").write_text(json.dumps(prediccion_nueva, indent=2))
else:
    print("Configura IMAGEN_NUEVA si deseas probar otra fotografía.")

## 14. Guardar, continuar e integrar la herramienta

El ZIP conserva configuración, catálogo, modelos completos, selección e historiales; excluye copias regenerables de datos y recortes. Para continuar el entrenamiento recupera **los outputs v3**, mantén parámetros y ejecuta de arriba abajo. Los estados y modelos v2 no son compatibles con esta cabeza.

Para usar la herramienta en una sesión nueva, ejecuta primero las definiciones de `UmbralesOrdenados`, `MAEOrdinal` y `decodificar_ordinal`, carga `clases_bcs.json` en `CLASES_BCS` y el modelo completo con `keras.models.load_model`. Los checkpoints pueden cargarse para inferencia con `compile=False`; igualmente necesitan la clase de capa registrada. Conserva las funciones de recorte y asociación, el detector y el mismo orden del catálogo. La celda de imagen nueva demuestra el flujo completo.

No se promete continuidad exacta si una sesión se interrumpe durante el guardado. La tasa del optimizador se conserva al reanudar; el contador del reductor se reinicia. La versión será candidata a uso final solo después de comprobar su desempeño y estabilidad en nuevos animales.

In [ ]:
import zipfile

ruta_zip = SALIDA.parent / (EXPERIMENTO + "_resultados.zip")
carpetas_regenerables = {"yolo_datos", "recortes_gt", "recortes_predichos"}
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in sorted(SALIDA.rglob("*")):
        relativa = ruta.relative_to(SALIDA)
        if ruta.is_file() and relativa.parts[0] not in carpetas_regenerables:
            archivo.write(ruta, str(Path(EXPERIMENTO) / relativa))
print("Archivo para descargar:", ruta_zip)
print(f"Tamaño: {ruta_zip.stat().st_size / 1024**2:.1f} MB")